# Notebook d'exploration 

In [90]:
import pandas as pd
import sys
sys.path.append('../src')
from tool import inspector
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## 1.1 Exploration des données 

In [91]:
df_customers = pd.read_csv(r"..\data\raw\customers.csv", sep=";")
df_products = pd.read_csv(r"..\data\raw\products.csv", sep=";")
df_transactions = pd.read_csv(r"..\data\raw\Transactions.csv", sep=";")

C:\Users\quent\AppData\Local\Temp\ipykernel_22948\430076701.py:3: DtypeWarning: Columns (0,1,2,3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_transactions = pd.read_csv(r"..\data\raw\Transactions.csv", sep=";")


In [92]:
inspector(df_customers)

######### DEBUT DE L'ANALYSE #########
Le df comprend 8621 lignes et 3 colonnes 

---------------------------------------------------------------
le df contient les nuls suivant 
client_id    0
sex          0
birth        0
dtype: int64

---------------------------------------------------------------
les types et noms de colonnes sont les suivants
client_id    object
sex          object
birth         int64
dtype: object

---------------------------------------------------------------
le nombre d'uniques dans les colonnes sont les suivants
client_id    8621
sex             2
birth          76
dtype: int64

---------------------------------------------------------------

######### FIN DE L'ANALYSE #########


In [93]:
inspector(df_products)

######### DEBUT DE L'ANALYSE #########
Le df comprend 3286 lignes et 3 colonnes 

---------------------------------------------------------------
le df contient les nuls suivant 
id_prod    0
price      0
categ      0
dtype: int64

---------------------------------------------------------------
les types et noms de colonnes sont les suivants
id_prod     object
price      float64
categ        int64
dtype: object

---------------------------------------------------------------
le nombre d'uniques dans les colonnes sont les suivants
id_prod    3286
price      1454
categ         3
dtype: int64

---------------------------------------------------------------

######### FIN DE L'ANALYSE #########


In [94]:
inspector(df_transactions)

######### DEBUT DE L'ANALYSE #########
Le df comprend 1048575 lignes et 4 colonnes 

---------------------------------------------------------------
le df contient les nuls suivant 
id_prod       361041
date          361041
session_id    361041
client_id     361041
dtype: int64

---------------------------------------------------------------
les types et noms de colonnes sont les suivants
id_prod       object
date          object
session_id    object
client_id     object
dtype: object

---------------------------------------------------------------
le nombre d'uniques dans les colonnes sont les suivants
id_prod         3265
date          687419
session_id    345505
client_id       8600
dtype: int64

---------------------------------------------------------------

######### FIN DE L'ANALYSE #########


## 1.2 Gestion des nulls

In [95]:
# Supprimer les lignes avec des données manquantes dans toutes les colonnes
df_transactions = df_transactions.dropna()
df_transactions.info()

<class 'pandas.core.frame.DataFrame'>
Index: 687534 entries, 0 to 687533
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   id_prod     687534 non-null  object
 1   date        687534 non-null  object
 2   session_id  687534 non-null  object
 3   client_id   687534 non-null  object
dtypes: object(4)
memory usage: 26.2+ MB


## 2.1  Merge des données

In [96]:
df_prod_transac = df_transactions.merge(df_products, on="id_prod", how="left")
print(f" ## merge de df_trasactions et df_products OK ##\n")
print(f" - ligne avant : {len(df_transactions):,}")
print(f" - ligne aprés {len(df_prod_transac):,}")
print(f" - colonne {list(df_prod_transac.columns)}\n")

df = df_prod_transac.merge(df_customers,on="client_id", how="left")
print(f" ## merge de df_prod_trasc et customers OK ##\n")
print(f" - ligne avant : {len(df):,}")
print(f" - ligne aprés {len(df):,}")
print(f" - colonne {list(df.columns)}")


 ## merge de df_trasactions et df_products OK ##

 - ligne avant : 687,534
 - ligne aprés 687,534
 - colonne ['id_prod', 'date', 'session_id', 'client_id', 'price', 'categ']

 ## merge de df_prod_trasc et customers OK ##

 - ligne avant : 687,534
 - ligne aprés 687,534
 - colonne ['id_prod', 'date', 'session_id', 'client_id', 'price', 'categ', 'sex', 'birth']


## 2.2 convertion de la colonne date au bon format 

In [97]:
#convertion des date au bon format
df['date'] = pd.to_datetime(df['date'])

In [98]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 687534 entries, 0 to 687533
Data columns (total 8 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   id_prod     687534 non-null  object        
 1   date        687534 non-null  datetime64[ns]
 2   session_id  687534 non-null  object        
 3   client_id   687534 non-null  object        
 4   price       687534 non-null  float64       
 5   categ       687534 non-null  int64         
 6   sex         687534 non-null  object        
 7   birth       687534 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 42.0+ MB


## 2.3 Analyse sur les clients B2B 

In [99]:
#chiffre d'affaire par client 

ca_par_client = df.groupby('client_id')['price'].sum().reset_index()

#tris pas chiffre d'affaire 
ca_par_client = ca_par_client.sort_values(by='price', ascending=False).reset_index(drop=True)
ca_par_client.head(10)

,client_id,price
0,c_1609,326039.89
1,c_4958,290227.03
2,c_6714,153918.60
3,c_3454,114110.57
4,c_1570,5285.82
5,c_3263,5276.87
6,c_2140,5260.18
7,c_2899,5214.05
8,c_7319,5155.77
9,c_7959,5135.75


In [100]:
client_b2b = ['c_1609', 'c4958', 'c6714','c3454']

df['segment_client'] = df['client_id'].apply(lambda x: 'B2B' if x in client_b2b else 'B2C')

df.head()

,id_prod,date,session_id,client_id,price,categ,sex,birth,segment_client
0,0_1259,2021-03-01 00:01:07.843138,s_1,c_329,11.99,0,f,1967,B2C
1,0_1390,2021-03-01 00:02:26.047414,s_2,c_664,19.37,0,m,1960,B2C
2,0_1352,2021-03-01 00:02:38.311413,s_3,c_580,4.50,0,m,1988,B2C
3,0_1458,2021-03-01 00:04:54.559692,s_4,c_7912,6.55,0,f,1989,B2C
4,0_1358,2021-03-01 00:05:18.801198,s_5,c_2033,16.49,0,f,1956,B2C


## 2.4 Ajout de la colonnes age des clients 

In [101]:
df['age'] = df['birth'].apply(lambda x: 2026 - x)

df.head()

,id_prod,date,session_id,client_id,price,categ,sex,birth,segment_client,age
0,0_1259,2021-03-01 00:01:07.843138,s_1,c_329,11.99,0,f,1967,B2C,59
1,0_1390,2021-03-01 00:02:26.047414,s_2,c_664,19.37,0,m,1960,B2C,66
2,0_1352,2021-03-01 00:02:38.311413,s_3,c_580,4.50,0,m,1988,B2C,38
3,0_1458,2021-03-01 00:04:54.559692,s_4,c_7912,6.55,0,f,1989,B2C,37
4,0_1358,2021-03-01 00:05:18.801198,s_5,c_2033,16.49,0,f,1956,B2C,70


# Export du fichier CSV

In [102]:
df.to_csv("df_b2b_age.csv", index=False)

# Calcul et graphique 
## 3.1 chiffre d'affaire et moyenne mobile 

In [ ]:
#grouper le CA par jour
ca_journalier = df.groupby(pd.Grouper(key='date', freq='D'))['price'].sum().reset_index()
ca_journalier.rename(columns={"price":"ca"}, inplace=True)
ca_journalier.head()

,date,ca
0,2021-03-01,16565.22
1,2021-03-02,15486.45
2,2021-03-03,15198.69
3,2021-03-04,15196.07
4,2021-03-05,17471.37


In [104]:
#trier par date (inspensable pour les moy mobile)
ca_journalier = ca_journalier.sort_values('date').reset_index(drop=True)
#moyenne mobile sur 7 jours
ca_journalier['mm_7j'] = ca_journalier['ca'].rolling(window=7).mean()
#moyenne mobile sur 10 jours 
ca_journalier['mm_10j'] = ca_journalier['ca'].rolling(window=10).mean()
#moyenne mobile sur 30 jours 
ca_journalier['mm_30j'] = ca_journalier['ca'].rolling(window=30).mean()

print("moyenne mobiles calculées")
print(ca_journalier.head(10))

moyenne mobiles calculées
        date        ca         mm_7j     mm_10j  mm_30j
0 2021-03-01  16565.22           NaN        NaN     NaN
1 2021-03-02  15486.45           NaN        NaN     NaN
2 2021-03-03  15198.69           NaN        NaN     NaN
3 2021-03-04  15196.07           NaN        NaN     NaN
4 2021-03-05  17471.37           NaN        NaN     NaN
5 2021-03-06  15785.28           NaN        NaN     NaN
6 2021-03-07  14760.20  15780.468571        NaN     NaN
7 2021-03-08  15679.53  15653.941429        NaN     NaN
8 2021-03-09  15710.51  15685.950000        NaN     NaN
9 2021-03-10  15496.87  15728.547143  15735.019     NaN


In [105]:
#Graphique des moyenne mobile 

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=ca_journalier['date'], 
    y=ca_journalier['ca'],
    mode='lines',
    name='CA Journalier',
    line=dict(width=0.3, color='gray') 
))
fig.add_trace(go.Scatter(
    x=ca_journalier['date'], 
    y=ca_journalier['mm_7j'],
    mode='lines',
    name='Moyenne Mobile 7j',
    line=dict(width=2.5, color='blue') 
))
fig.add_trace(go.Scatter(
    x=ca_journalier['date'], 
    y=ca_journalier['mm_10j'],
    mode='lines',
    name='Moyenne Mobile 10j',
    line=dict(width=3, color='green') 
))
fig.add_trace(go.Scatter(
    x=ca_journalier['date'], 
    y=ca_journalier['mm_30j'],
    mode='lines',
    name='Moyenne Mobile 30j',
    line=dict(width=4.5, color='red') 
))
# mise en forme
fig.update_layout(
    title="Analyse du CA et Moyennes Mobiles",
    xaxis_title="Date",
    yaxis_title="Montant (€)",
    template="plotly_white"
)

fig.show()

## chiffre d'affaire par catégories 

In [106]:
ca_categorie = df.groupby(by="categ")['price'].sum().reset_index()

ca_categorie.head()

,categ,price
0,0,4419730.97
1,1,4827657.11
2,2,2780275.02


In [107]:
fig = go.Figure()
fig.add_trace(go.Bar(
    x=ca_categorie['categ'], 
    y=ca_categorie['price'],
    name='CA par catégories',
    marker_color=['blue', 'green', 'orange']
))
# mise en forme
fig.update_layout(
    title_text='Répartition du CA par catégorie',
    xaxis=dict(type='category')
)

fig.show()

In [108]:
#grouper les clients par mois
ca_mensuel_categ = df.groupby([pd.Grouper(key='date', freq='M'), 'categ'])['price'].sum().reset_index()
ca_mensuel_categ.head()

C:\Users\quent\AppData\Local\Temp\ipykernel_22948\1212606156.py:2: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  ca_mensuel_categ = df.groupby([pd.Grouper(key='date', freq='M'), 'categ'])['price'].sum().reset_index()


,date,categ,price
0,2021-03-31,0,193629.17
1,2021-03-31,1,186974.17
2,2021-03-31,2,101837.27
3,2021-04-30,0,205222.46
4,2021-04-30,1,156138.35


In [109]:
#Graphique des moyenne mobile 

fig = go.Figure()


for c in ca_mensuel_categ['categ'].unique():
    # On isole les données de la catégorie
    data_filtree = ca_mensuel_categ[ca_mensuel_categ['categ'] == c]
    
    fig.add_trace(go.Scatter(
        x=data_filtree['date'],
        y=data_filtree['price'],
        mode='lines', 
        name=f"Catégorie {c}",
        connectgaps=True, # Au cas où un mois n'aurait pas de données
        hovertemplate = "%{y:.0f}€"
    ))


# mise en forme
fig.update_layout(
    title="Évolution du CA mensuel par catégorie",
    xaxis_title="Mois",
    yaxis_title="CA (€)",
    hovermode="x unified" # affiche toutes les valeurs au même point X au survol
    
)

fig.show()

## Nombre de clients par mois 

In [110]:
#grouper les clients par mois
client_mois = df.groupby(pd.Grouper(key='date', freq='M'))['client_id'].nunique().reset_index()
client_mois.rename(columns={"client_id":"client"}, inplace=True)
client_mois.head()

C:\Users\quent\AppData\Local\Temp\ipykernel_22948\91020028.py:2: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  client_mois = df.groupby(pd.Grouper(key='date', freq='M'))['client_id'].nunique().reset_index()


,date,client
0,2021-03-31,5676
1,2021-04-30,5674
2,2021-05-31,5644
3,2021-06-30,5659
4,2021-07-31,5672


In [111]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=client_mois['date'], 
    y=client_mois['client'],
    mode='lines',
    name='client par mois',
    line=dict(width=2, color='red') 
))

# mise en forme
fig.update_layout(
    title="Évolution des clients mensuel",
    xaxis_title="Mois",
    yaxis_title="Nombre de client"
)
fig.show()

## Nombre de transactions

In [112]:
#grouper les transaction par jour/semaines/mois
transac_jour = df.groupby(pd.Grouper(key='date', freq='D'))['session_id'].nunique().reset_index()
transac_jour.rename(columns={"session_id":"Transaction"}, inplace=True)

transac_semaine = df.groupby(pd.Grouper(key='date', freq='W'))['session_id'].nunique().reset_index()
transac_semaine.rename(columns={"session_id":"Transaction"}, inplace=True)

transac_mois = df.groupby(pd.Grouper(key='date', freq='M'))['session_id'].nunique().reset_index()
transac_mois.rename(columns={"session_id":"Transaction"}, inplace=True)


C:\Users\quent\AppData\Local\Temp\ipykernel_22948\3518447641.py:8: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  transac_mois = df.groupby(pd.Grouper(key='date', freq='M'))['session_id'].nunique().reset_index()


In [113]:
#moyenne transaction journaliere 

moyenne_transac_jours = transac_jour['Transaction'].mean()

print(f"la moyenne de transaction par jours est {moyenne_transac_jours:.2f}")

#moyenne transaction hebdomadaire
moyenne_transac_semaine = transac_semaine['Transaction'].mean()

print(f"la moyenne de transaction par semaines est {moyenne_transac_semaine:.2f}")

#moyenne transaction mensuel
moyenne_transac_mois = transac_mois['Transaction'].mean()

print(f"la moyenne de transaction par mois est {moyenne_transac_mois:.2f}")

la moyenne de transaction par jours est 475.47
la moyenne de transaction par semaines est 3292.72
la moyenne de transaction par mois est 14398.17


In [114]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=transac_mois['date'], 
    y=transac_mois['Transaction'],
    mode='lines',
    name='Transaction par mois',
    line=dict(width=2, color='red') ,
    hovertemplate= "%{y:,.2f} €"
))
fig.update_layout(
    title="Évolution des transactions mensuel",
    xaxis_title="Mois",
    yaxis_title="Nombre de transactions",
    hovermode = "x unified",
    separators = ". "
)
fig.show()

## Nombres de produits vendus 

In [115]:
stats_produits = df.groupby(pd.Grouper(key='date', freq='M')).agg({
    'id_prod': ['count', 'nunique'] # On demande les deux d'un coup !
}).reset_index()

# On simplifie les noms de colonnes qui sont en multi-index
stats_produits.columns = ['date', 'volume_total', 'references_uniques']

C:\Users\quent\AppData\Local\Temp\ipykernel_22948\2478061302.py:1: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  stats_produits = df.groupby(pd.Grouper(key='date', freq='M')).agg({


In [116]:
# 1. On crée une figure avec un axe Y secondaire
fig = make_subplots(specs=[[{"secondary_y": True}]])

# 2. Ajout du Volume (Barres)
fig.add_trace(
    go.Bar(
        x=stats_produits['date'], 
        y=stats_produits['volume_total'],
        name="Volume total vendu",
        marker_color='lightblue',
        opacity=0.6
    ),
    secondary_y=False,
)

# 3. Ajout de la Diversité (Ligne)
fig.add_trace(
    go.Scatter(
        x=stats_produits['date'], 
        y=stats_produits['references_uniques'],
        name="Réf. uniques vendues",
        mode='lines+markers',
        line=dict(color='red', width=3)
    ),
    secondary_y=True,
)

# 4. On nomme les axes pour que ce soit clair
fig.update_layout(
    title_text="Analyse de la performance du catalogue",
    xaxis_title="Mois",
    hovermode = "x unified"

)

fig.update_yaxes(title_text="<b>Volume</b> (nb ventes)", secondary_y=False)
fig.update_yaxes(title_text="<b>Diversité</b> (nb réf. uniques)", secondary_y=True)

fig.show()

In [117]:
#grouper les produits vendus par mois
produits_mois = df.groupby(pd.Grouper(key='date', freq='M'))['id_prod'].nunique().reset_index()
produits_mois.rename(columns={"id_prod":"Produits"}, inplace=True)
produits_mois.head()

C:\Users\quent\AppData\Local\Temp\ipykernel_22948\39758037.py:2: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  produits_mois = df.groupby(pd.Grouper(key='date', freq='M'))['id_prod'].nunique().reset_index()


,date,Produits
0,2021-03-31,2482
1,2021-04-30,2492
2,2021-05-31,2471
3,2021-06-30,2414
4,2021-07-31,2369
